# Required Security Workflow — Image Prototype

This notebook arranges the existing project code in the sequence suggested by the assignment brief. Relevant protocol code is included directly from `payload_protocol.py`. It does not add implementations for requirements that are not yet present.


In [ ]:
import hashlib
import json
import secrets
import struct
from datetime import datetime, timezone

import numpy as np
from cryptography.exceptions import InvalidSignature
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.asymmetric import padding, rsa


RSA_KEY_SIZE = 2048

MEDIA_PREFIXES = {
    "image": "IMG",
    "audio": "AUD",
}


class UnSupportedFileType(Exception):
    pass


## 1. Select an original cover object

The existing FR1 code accepts a PNG path, validates the file, and returns an RGB `uint8` array. The protocol also detects whether supplied cover bytes represent PNG or WAV media.


In [ ]:
def load_png_from_path(image_path):
    from os import PathLike
    from PIL import Image

    if not isinstance(image_path, (str, bytes, PathLike)):
        raise TypeError("image_path must be a filesystem path")
    
    try:
        with Image.open(image_path) as image:
            if image.format != "PNG":
                raise UnSupportedFileType(
                    f"Unsupported file type: {image.format or 'unknown'}"
                )
            
            image.load()
            if image.width < 1 or image.height < 1:
                raise ValueError("Image dimensions must be greater than zero")

            image_array = np.array(image.convert("RGB"), dtype=np.uint8, copy=True)
            if image_array.ndim != 3 or image_array.shape[2] != 3:
                raise ValueError("Decoded image must have three colour channels")
            
            return image_array
        
    except UnSupportedFileType:
        raise
    
    except (OSError, ValueError) as error:
        raise ValueError("Unreadable or corrupted PNG image") from error


In [ ]:
def detect_media_type(cover_bytes: bytes) -> str:
    """
    Detect whether the supplied bytes represent a PNG image
    or a WAV audio file.

    Returns:
        "image" for PNG
        "audio" for WAV

    Raises:
        ValueError for unsupported media formats.
    """

    if not isinstance(cover_bytes, bytes):
        raise TypeError("cover_bytes must be bytes")

    # PNG files start with this 8-byte signature.
    png_signature = b"\x89PNG\r\n\x1a\n"

    if cover_bytes.startswith(png_signature):
        return "image"

    # Standard WAV files use a RIFF container:
    # bytes 0-3  = RIFF
    # bytes 8-11 = WAVE
    if (
        len(cover_bytes) >= 12
        and cover_bytes[0:4] == b"RIFF"
        and cover_bytes[8:12] == b"WAVE"
    ):
        return "audio"

    raise ValueError(
        "Unsupported media format. Expected PNG image or WAV audio."
    )


## 2. Compute a cryptographic hash

`calculate_media_hash` computes and returns the SHA-256 hash of the original cover bytes.


In [ ]:
def calculate_media_hash(cover_bytes: bytes) -> str:
    """Return the SHA-256 hexadecimal digest of the cover bytes."""

    if not isinstance(cover_bytes, bytes):
        raise TypeError("cover_bytes must be bytes")

    return hashlib.sha256(cover_bytes).hexdigest()


## 3. Create the verification payload

`create_payload` receives the previously calculated hash and places it in the compact JSON payload.


In [ ]:
def create_payload(
    cover_bytes: bytes,
    media_hash: str,
    team_id: str,
    sender: str,
) -> bytes:
    """
    Create a compact verification payload.

    FR3 required fields:
    - media_id
    - timestamp
    - media_hash
    - nonce
    - team-defined metadata

    Additional field:
    - media_type, automatically detected as "image" or "audio"

    The returned payload is UTF-8 encoded compact JSON bytes.
    """

    if not isinstance(cover_bytes, bytes):
        raise TypeError("cover_bytes must be bytes")

    if not isinstance(media_hash, str):
        raise TypeError("media_hash must be a string")

    if not isinstance(team_id, str):
        raise TypeError("team_id must be a string")

    if not isinstance(sender, str):
        raise TypeError("sender must be a string")

    media_hash = media_hash.strip().lower()
    team_id = team_id.strip()
    sender = sender.strip()

    if (
        len(media_hash) != 64
        or any(character not in "0123456789abcdef" for character in media_hash)
    ):
        raise ValueError("media_hash must be a SHA-256 hexadecimal digest")

    if not team_id:
        raise ValueError("team_id cannot be empty")

    if not sender:
        raise ValueError("sender cannot be empty")

    # Automatically determine whether the cover is PNG or WAV.
    media_type = detect_media_type(cover_bytes)

    # Auto-generate a media ID.
    # Example: IMG-a3f92c10 or AUD-51bc1234
    prefix = MEDIA_PREFIXES[media_type]
    media_id = f"{prefix}-{secrets.token_hex(4)}"

    timestamp = datetime.now(timezone.utc).strftime(
        "%Y-%m-%dT%H:%M:%SZ"
    )

    # 16 random bytes = 128-bit nonce.
    nonce = secrets.token_hex(16)

    # Team-defined metadata.
    metadata = {
        "team_id": team_id,
        "sender": sender,
    }

    payload = {
        "media_id": media_id,
        "media_type": media_type,
        "timestamp": timestamp,
        "media_hash": media_hash,
        "nonce": nonce,
        "metadata": metadata,
    }

    return json.dumps(
        payload,
        sort_keys=True,
        separators=(",", ":"),
        allow_nan=False,
    ).encode("utf-8")


## 4. Sign the payload

The existing protocol generates an RSA key pair and signs the exact payload bytes.


In [ ]:
def generate_rsa_keypair(
) -> tuple[rsa.RSAPrivateKey, rsa.RSAPublicKey]:
    """
    Generate an RSA-2048 private/public key pair.

    Private key -> signing
    Public key  -> verification
    """

    private_key = rsa.generate_private_key(
        public_exponent=65537,
        key_size=RSA_KEY_SIZE,
    )

    return private_key, private_key.public_key()


def _get_pss_padding() -> padding.PSS:
    """
    Return the RSA-PSS padding configuration used by both
    signing and verification.
    """

    return padding.PSS(
        mgf=padding.MGF1(hashes.SHA256()),
        salt_length=padding.PSS.MAX_LENGTH,
    )


def sign_payload(
    payload_bytes: bytes,
    private_key: rsa.RSAPrivateKey,
) -> bytes:
    """
    Digitally sign the exact FR3 payload bytes.

    Algorithm:
    - RSA-2048
    - RSA-PSS padding
    - SHA-256
    """

    if not isinstance(payload_bytes, bytes):
        raise TypeError("payload_bytes must be bytes")

    if not isinstance(private_key, rsa.RSAPrivateKey):
        raise TypeError(
            "private_key must be an RSA private key"
        )

    return private_key.sign(
        payload_bytes,
        _get_pss_padding(),
        hashes.SHA256(),
    )


## 5. Select a start location and embed the packet

The protocol combines the payload and signature into one packet. The existing FR5 function embeds arbitrary bytes with one-bit LSB replacement. It currently starts at the first image channel and does not yet accept a selectable start location.


In [ ]:
def build_verification_packet(
    payload_bytes: bytes,
    signature: bytes,
) -> bytes:
    """
    Combine payload and signature into one byte packet.

    Format:
        4-byte big-endian payload length
        + payload bytes
        + RSA signature

    For RSA-2048, the signature is 256 bytes.
    """

    if not isinstance(payload_bytes, bytes):
        raise TypeError("payload_bytes must be bytes")

    if not isinstance(signature, bytes):
        raise TypeError("signature must be bytes")

    header = struct.pack(
        ">I",
        len(payload_bytes),
    )

    return header + payload_bytes + signature


In [ ]:
def embed_image_payload(img_array, payload: bytes):
    if not isinstance(img_array, np.ndarray):
        raise TypeError("img_array must be a numpy array")
    
    if img_array.dtype != np.uint8:
        raise TypeError("img_array must have dtype uint8")
    
    if not isinstance(payload, bytes):
        raise TypeError("payload must be bytes")

    payload_bits = np.unpackbits(np.frombuffer(payload, dtype=np.uint8))
    flat_img = img_array.flatten()
    if payload_bits.size > flat_img.size:
        raise ValueError("payload is too large for the image capacity")

    flat_img[:payload_bits.size] &= np.uint8(0xFE)
    flat_img[:payload_bits.size] |= payload_bits
    return flat_img.reshape(img_array.shape)


## 6. Output a stego image

The removed demonstration cell saved a sample image, but the project does not currently contain a reusable stego-image output function.


## 7. Identify the start location and extract the hidden packet

Image LSB extraction is not implemented yet. Once bytes have been extracted, the existing protocol can separate the packet into payload and signature bytes and decode the payload JSON.


In [ ]:
def unpack_verification_packet(
    raw_packet: bytes,
    public_key: rsa.RSAPublicKey,
) -> tuple[bytes, bytes]:
    """
    Extract payload bytes and signature bytes from a packet.

    Extra trailing bytes are ignored because steganography
    extraction may return a larger buffer.
    """

    if not isinstance(raw_packet, bytes):
        raise TypeError("raw_packet must be bytes")

    if not isinstance(public_key, rsa.RSAPublicKey):
        raise TypeError(
            "public_key must be an RSA public key"
        )

    signature_length = (
        public_key.key_size + 7
    ) // 8

    minimum_size = 4 + signature_length

    if len(raw_packet) < minimum_size:
        raise ValueError(
            "Payload missing or incomplete"
        )

    payload_length = struct.unpack(
        ">I",
        raw_packet[:4],
    )[0]

    payload_start = 4
    payload_end = (
        payload_start + payload_length
    )

    signature_end = (
        payload_end + signature_length
    )

    if len(raw_packet) < signature_end:
        raise ValueError(
            "Payload missing or corrupted"
        )

    payload_bytes = raw_packet[
        payload_start:payload_end
    ]

    signature = raw_packet[
        payload_end:signature_end
    ]

    return payload_bytes, signature


def decode_payload(payload_bytes: bytes) -> dict:

    if not isinstance(payload_bytes, bytes):
        raise TypeError("payload_bytes must be bytes")

    try:
        payload = json.loads(payload_bytes.decode("utf-8"))

    except (UnicodeDecodeError, json.JSONDecodeError) as error:
        raise ValueError("Payload is not valid JSON") from error

    if not isinstance(payload, dict):
        raise ValueError("Payload must contain a JSON object")

    return payload


## 8. Verify the signature

The existing protocol verifies the extracted payload bytes and signature with the RSA public key.


In [ ]:
def verify_signature(
    payload_bytes: bytes,
    signature: bytes,
    public_key: rsa.RSAPublicKey,
) -> bool:
    """
    Verify the digital signature using the corresponding
    RSA public key.

    Returns:
        True  -> signature valid
        False -> signature invalid
    """

    if not isinstance(payload_bytes, bytes):
        raise TypeError("payload_bytes must be bytes")

    if not isinstance(signature, bytes):
        raise TypeError("signature must be bytes")

    if not isinstance(public_key, rsa.RSAPublicKey):
        raise TypeError(
            "public_key must be an RSA public key"
        )

    try:
        public_key.verify(
            signature,
            payload_bytes,
            _get_pss_padding(),
            hashes.SHA256(),
        )

        return True

    except InvalidSignature:
        return False


## 9. Recompute the current media hash

Current-media hash comparison is not implemented. `create_payload` only records the hash used when the payload is created.


## 10. Return a verdict

The existing packet verifier reports packet, signature, and JSON-decoding results. It intentionally does not perform the FR9 media-hash comparison.


In [ ]:
def verify_verification_packet(
    raw_packet: bytes,
    public_key: rsa.RSAPublicKey,
) -> tuple[bool, str, dict | None]:
    """
    Verify a complete payload/signature packet.

    Performs:
    1. packet extraction
    2. FR4 signature verification
    3. payload JSON decoding

    This function intentionally does NOT perform FR9
    media-hash verification.
    """

    try:
        payload_bytes, signature = (
            unpack_verification_packet(
                raw_packet,
                public_key,
            )
        )

    except ValueError as error:
        return False, str(error), None

    if not verify_signature(
        payload_bytes,
        signature,
        public_key,
    ):
        return (
            False,
            "Signature Invalid",
            None,
        )

    try:
        payload = decode_payload(
            payload_bytes
        )

    except ValueError:
        return (
            False,
            "Payload Missing or Corrupted",
            None,
        )

    return (
        True,
        "Signature Valid",
        payload,
    )
